# KRM-Core: Hugging Face + RWKV Training

Kavramsal Parçalı Eğitim ile HF veri setlerinden model eğitimi.

**Desteklenen veri setleri:**
- `bigcode/the-stack` (kod, 6TB+)
- `HuggingFaceFW/fineweb` (genel bilgi, 15T token)
- `codeparrot/github-code` (GitHub kodu)
- `tiiuae/falcon-refinedweb` (web metni)
- `allenai/c4` (temiz web)
- Kendi özel verin (Drive)

In [ ]:
# Bağımlılıklar
!pip install torch numpy tqdm datasets huggingface_hub

import torch, gc, json, time, os, random
from pathlib import Path
from datasets import load_dataset, get_dataset_config_names, load_from_disk
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# RWKV Model
class WKVMemory(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.log_gain = nn.Parameter(torch.zeros(d_model))
        self.log_decay = nn.Parameter(torch.zeros(d_model))

    def forward(self, values, keys):
        B, S, D = values.shape
        gain = torch.exp(self.log_gain)
        decay = torch.exp(-torch.exp(self.log_decay))
        out = torch.zeros_like(values)
        s_c = torch.zeros(B, D, device=values.device)
        s_n = torch.zeros(B, D, device=values.device)
        for t in range(S):
            k = keys[:, t] * gain
            s_c = s_c * decay + k * values[:, t]
            s_n = s_n * decay + k
            out[:, t] = s_c / (s_n + 1e-8)
        return out


class ConceptBlock(nn.Module):
    def __init__(self, block_id, vocab_size=256, d_model=256, n_layers=2, d_ff=1024):
        super().__init__()
        self.block_id = block_id
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.time_mix = nn.ModuleList([
            nn.ModuleDict({'k': nn.Linear(d_model,d_model,0), 'v': nn.Linear(d_model,d_model,0),
                           'r': nn.Linear(d_model,d_model,0), 'o': nn.Linear(d_model,d_model,0),
                           'wkv': WKVMemory(d_model)})
            for _ in range(n_layers)])
        self.chan_mix = nn.ModuleList([
            nn.ModuleDict({'k': nn.Linear(d_model,d_ff,0), 'v': nn.Linear(d_ff,d_model,0),
                           'r': nn.Linear(d_model,d_model,0)})
            for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size, False)

    def forward(self, x, targets=None):
        h = self.embedding(x)
        for tm, cm in zip(self.time_mix, self.chan_mix):
            r = h
            n = self.ln1(h)
            k, v, r_g = tm['k'](n), tm['v'](n), tm['r'](n)
            h = r + torch.sigmoid(r_g) * tm['o'](tm['wkv'](v, k))
            r = h
            n = self.ln2(h)
            k = F.relu(cm['k'](n)) ** 2
            h = r + torch.sigmoid(cm['r'](n)) * cm['v'](k)
        logits = self.out(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def get_param_count(self):
        return sum(p.numel() for p in self.parameters())

In [ ]:
# Tokenizer
class CharTokenizer:
    def __init__(self):
        self.vocab_size = 256
        self.bos, self.eos, self.pad, self.unk = 1, 2, 0, 3
    def encode(self, text):
        return [self.bos] + [(ord(c) if ord(c) < 256 else self.unk) for c in text[:100000]] + [self.eos]
    def decode(self, tokens):
        return ''.join(chr(t) for t in tokens if 4 <= t < 256)

tokenizer = CharTokenizer()

In [ ]:
# ============================================================
# HUGGING FACE VERI SETI SECIMI
# ============================================================

# Hangi veri seti kullanılacak?
# Seçenek 1: FineWeb (genel bilgi, 15T token)
# Seçenek 2: The Stack (kod, 6TB)
# Seçenek 3: C4 (temiz web)

DATASET_CHOICE = "fineweb"  # fineweb, the-stack, c4
SAMPLE_SIZE = 50000  # Kac ornek kullanilacak

print(f"Veri seti: {DATASET_CHOICE}")
print(f"Ornek sayisi: {SAMPLE_SIZE:,}")

if DATASET_CHOICE == "fineweb":
    print("\nFineWeb indiriliyor... (10M ornek = ~1GB)")
    ds = load_dataset("HuggingFaceFW/fineweb", split="train", streaming=False)[:SAMPLE_SIZE]
    texts = [d['text'] for d in ds if d.get('text')]

elif DATASET_CHOICE == "the-stack":
    print("\nThe Stack indiriliyor... (Python kodu)")
    ds = load_dataset("bigcode/the-stack", data_dir="data/python",
                      split="train", streaming=False)[:SAMPLE_SIZE]
    texts = [d['content'] for d in ds if d.get('content')]

elif DATASET_CHOICE == "c4":
    print("\nC4 (temiz web) indiriliyor...")
    ds = load_dataset("allenai/c4", "en", split="train", streaming=False)[:SAMPLE_SIZE]
    texts = [d['text'] for d in ds if d.get('text')]

print(f"Toplam: {len(texts):,} ornek")
total_chars = sum(len(t) for t in texts)
print(f"Karakter: {total_chars:,} (~{total_chars//4:,} token)")

# Cache'e kaydet (tekrar indirme)
Path('hf_data').mkdir(exist_ok=True)
with open('hf_data/corpus.txt', 'w', encoding='utf-8') as f:
    for t in texts:
        f.write(t.strip() + '\n\n---\n\n')

In [ ]:
# Dataset
class HFDataset(Dataset):
    def __init__(self, corpus_path, seq_len=256, max_samples=10000):
        self.seq_len = seq_len
        with open(corpus_path, 'r') as f:
            text = f.read()
        tokens = tokenizer.encode(text)
        stride = max(1, (len(tokens) - seq_len) // max_samples)
        self.samples = []
        for i in range(0, len(tokens) - seq_len, stride):
            if len(self.samples) >= max_samples:
                break
            chunk = tokens[i:i+seq_len]
            if len(chunk) < seq_len:
                chunk = chunk + [0] * (seq_len - len(chunk))
            self.samples.append((
                torch.tensor(chunk[:-1], dtype=torch.long),
                torch.tensor(chunk[1:], dtype=torch.long)
            ))
        print(f"Samples: {len(self.samples):,}")
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
# ============================================================
# EGITIM
# ============================================================

CONFIG = {
    'd_model': 384,
    'n_layers': 2,
    'd_ff': 1536,
    'seq_len': 256,
    'batch_size': 32,
    'epochs': 10,
    'lr': 3e-4,
    'max_samples': 50000,
    'num_blocks': 1,
}

print("Config:", json.dumps(CONFIG, indent=2))

# Veri
dataset = HFDataset('hf_data/corpus.txt', CONFIG['seq_len'], CONFIG['max_samples'])
val_size = int(len(dataset) * 0.05)
train_ds, val_ds = torch.utils.data.random_split(dataset, [len(dataset)-val_size, val_size])
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'])

# Model
block = ConceptBlock('hf_block', d_model=CONFIG['d_model'],
                     n_layers=CONFIG['n_layers'], d_ff=CONFIG['d_ff'])
block.to(device)
params = block.get_param_count()
print(f"\nParams: {params:,} ({params*4/1024/1024:.1f} MB)")

optim = torch.optim.AdamW(block.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, CONFIG['epochs'])

history = {'train_loss': [], 'val_loss': [], 'val_ppl': []}
best_loss = float('inf')

print("\n" + "="*50)
print("TRAINING")
print("="*50)

for epoch in range(CONFIG['epochs']):
    t0 = time.time()

    # Train
    block.train()
    total_loss, total_tokens = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        _, loss = block(x, y)
        optim.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(block.parameters(), 1.0)
        optim.step()
        total_loss += loss.item()
        total_tokens += y.numel()

    # Val
    block.eval()
    val_loss, val_tokens = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = block(x, y)
            val_loss += loss.item()
            val_tokens += y.numel()

    avg_train = total_loss / total_tokens
    avg_val = val_loss / val_tokens
    ppl = torch.exp(torch.tensor(avg_val)).item()

    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    history['val_ppl'].append(ppl)

    print(f"Ep {epoch+1:2d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | PPL: {ppl:.2f} | {time.time()-t0:.1f}s")

    if avg_val < best_loss:
        best_loss = avg_val
        torch.save({
            'state': block.state_dict(),
            'config': CONFIG,
            'history': history
        }, 'best_hf_model.pt')
        print(f"  [*] Kaydedildi!")

    scheduler.step()

print(f"\n{'='*50}")
print(f"EN IYI: Val Loss: {best_loss:.4f}, PPL: {torch.exp(torch.tensor(best_loss)).item():.2f}")
print(f"TOPLAM: {params:,} parametre")
print(f"VERI:   {CONFIG['max_samples']:,} ornek, {CONFIG['epochs']} epoch")

In [ ]:
# Örnek üretim
block.eval()
seed = torch.randint(4, 256, (1, 1), device=device)
gen = seed.tolist()[0]

with torch.no_grad():
    for _ in range(500):
        logits, _ = block(seed, None)
        p = F.softmax(logits[0, -1] / 0.8, dim=0)
        t = torch.multinomial(p, 1).item()
        gen.append(t)
        seed = torch.cat([seed, torch.tensor([[t]], device=device)], dim=1)[:, -CONFIG['seq_len']:]

print("--- URETILEN METIN ---")
print(tokenizer.decode(gen)[:1000])

In [ ]:
# Drive'a kaydet
from google.colab import drive
drive.mount('/content/drive')

save_dir = Path('/content/drive/MyDrive/KRM-Core')
save_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    'state': block.state_dict(),
    'config': CONFIG,
    'history': history,
    'best_loss': best_loss,
    'params': params,
    'dataset': DATASET_CHOICE,
    'samples': CONFIG['max_samples']
}, save_dir / 'hf_trained_model.pt')

import shutil
for f in ['best_hf_model.pt']:
    if Path(f).exists():
        shutil.copy(f, save_dir / f)

print(f"Model: {save_dir}/")
for f in save_dir.glob('*'):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f}MB)")

## Çoklu Blok Eğitimi

Her veri seti türü için ayrı bir ConceptBlock eğitilebilir:
- Blok 1: FineWeb (genel bilgi)
- Blok 2: The Stack (kod)
- Blok 3: arXiv (bilimsel)

Kavramsal Parçalı Eğitim ile tüm bloklar ayrı ayrı eğitilir,
inference'da Rezonans Motoru ile birleştirilir.